# Qwen2.5 Model Re-Routing Estimator Convergence Experiment

This notebook is the model re-routing estimator convergence experiment. The requested model configuration supplies the canary prompts, and those same canary prompts are used for all three actual model configurations.

For each ordered pair `(A, B)`:

- `A` supplies the canary prompt slots `C_1`, `C_2 = C_3`, and `C_4`.
- The requested model configuration supplies logits on those requested-configuration canary prompts for the estimator.
- The actual model configuration supplies random-decoding-altered sampling distributions on those same requested-configuration canary prompts.
- If top-p truncation removes token ID `15` or `16`, the curve is marked undefined.


## 1. Place Files

This notebook uses the saved model re-routing prompt-selection payloads. It will search for `logit_payloads/` from this folder, or for `model_routing_experiment/logit_payloads/` from the repo root.

Expected payload examples:

```text
model_routing_experiment/logit_payloads/qwen25_7b_8bit_prompt_1_logits.pt
model_routing_experiment/logit_payloads/qwen25_14b_8bit_prompt_4_logits.pt
model_routing_experiment/logit_payloads/qwen25_32b_4bit_prompt_6_logits.pt
```


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (path for path in PROJECT_ROOT_CANDIDATES if (path / "logit_helpers.py").exists() and (path / "estimators.py").exists()),
    None,
)
assert PROJECT_ROOT is not None, "Could not find repo root containing logit_helpers.py and estimators.py."
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PAYLOAD_DIR_CANDIDATES = [
    Path.cwd() / "logit_payloads",
    Path.cwd() / "model_routing_experiment" / "logit_payloads",
    PROJECT_ROOT / "model_routing_experiment" / "logit_payloads",
]
PAYLOAD_DIR = next((path for path in PAYLOAD_DIR_CANDIDATES if path.exists()), None)
assert PAYLOAD_DIR is not None, "Missing logit_payloads directory. Expected it in ./logit_payloads or ./model_routing_experiment/logit_payloads."
print("Using project root:", PROJECT_ROOT)
print("Using payload directory:", PAYLOAD_DIR)

import numpy as np
import torch

from logit_helpers import make_random_decoding_params
from logit_helpers import make_random_decoding_probability_vector
from logit_helpers import make_cumulative_probability_vector
from estimators import sample_g_for_fixed_iterations
from estimators import naive_method_i_estimator
from estimators import naive_method_ii_estimator

print("Ready. Payload files found:", len(list(PAYLOAD_DIR.glob("*.pt"))))


Using project root: C:\Users\beatn\LLM-Ownership-Identification-Draft-1
Using payload directory: C:\Users\beatn\LLM-Ownership-Identification-Draft-1\model_routing_experiment\logit_payloads


Ready. Payload files found: 18


## 2. Experiment Configuration

The token pair, random decoding seed, and canary prompt maps define the model re-routing requested-configuration-canary protocol. The requested model configuration supplies the canary prompt IDs for every actual config.

Each context runs for exactly `7,500` one-token samples unless a target token has zero probability after decoding. If a target token has zero probability after decoding, the corresponding run is marked undefined. The estimator treats context traces in parallel, while the graph x-axis reports total one-token calls across the three contexts.


In [2]:
TOKEN_A = 15  # decoded as "0"
TOKEN_B = 16  # decoded as "1"
INTEREST_TOKENS = (TOKEN_A, TOKEN_B)

FIXED_SAMPLES_PER_CONTEXT = 7_500

SHARED_DECODING_SEED = 20260722
SAMPLING_SEED_BASE = 910000

MODEL_ORDER = {"7b": 0, "14b": 1, "32b": 2}

MODELS = {
    "7b": {
        "label": "Qwen2.5-7B-Instruct",
        "prefix": "qwen25_7b_8bit",
        "canaries": {1: 1, 2: 2, 4: 3},
    },
    "14b": {
        "label": "Qwen2.5-14B-Instruct",
        "prefix": "qwen25_14b_8bit",
        "canaries": {1: 1, 2: 4, 4: 5},
    },
    "32b": {
        "label": "Qwen2.5-32B-Instruct",
        "prefix": "qwen25_32b_4bit",
        "canaries": {1: 2, 2: 6, 4: 4},
    },
}

REQUESTED_MODEL_CONFIG_KEYS = ["7b", "14b", "32b"]
ACTUAL_MODEL_CONFIG_KEYS = ["7b", "14b", "32b"]


## 3. Loading and Simulation Helpers

The helper names make the canary ownership explicit: `requested_key` chooses the prompt IDs; `actual_key` chooses the model distribution being sampled on those prompt IDs.


In [3]:
def payload_path(model_key: str, prompt_id: int) -> Path:
    prefix = MODELS[model_key]["prefix"]
    return PAYLOAD_DIR / f"{prefix}_prompt_{prompt_id}_logits.pt"


def load_logits(model_key: str, prompt_id: int) -> np.ndarray:
    path = payload_path(model_key, prompt_id)
    if not path.exists():
        raise FileNotFoundError(path)
    try:
        payload = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        payload = torch.load(path, map_location="cpu")
    return payload["logits"].detach().cpu().numpy().astype(np.float64, copy=False)


def requested_config_canary_prompts(requested_key: str) -> dict[int, int]:
    """Return the canary prompt map supplied by requested model configuration."""
    return dict(MODELS[requested_key]["canaries"])


def build_requested_z(requested_key: str) -> dict[tuple[int, int], float]:
    """Build z from requested-config logits on requested-config-supplied canary prompts."""
    z = {}
    for slot, prompt_id in requested_config_canary_prompts(requested_key).items():
        logits = load_logits(requested_key, prompt_id)
        z[(slot, 0)] = float(logits[TOKEN_A])
        z[(slot, 1)] = float(logits[TOKEN_B])
    return z


def category_probabilities(cdf) -> dict[int, float]:
    edges = np.concatenate(([0.0], np.asarray(cdf.cumulative_probs, dtype=np.float64)))
    masses = np.diff(edges)
    return {int(category): float(mass) for category, mass in zip(cdf.categories, masses)}


def target_token_status(probs_by_category: dict[int, float]) -> str:
    missing = []
    if probs_by_category.get(TOKEN_A, 0.0) <= 0.0:
        missing.append(f"token {TOKEN_A}")
    if probs_by_category.get(TOKEN_B, 0.0) <= 0.0:
        missing.append(f"token {TOKEN_B}")
    if missing:
        return "undefined: " + ", ".join(missing) + " has zero probability after decoding"
    return "sampled"


def build_actual_cdfs_on_requested_canaries(requested_key: str, actual_key: str, params) -> dict[int, object]:
    """Build B sampling CDFs on the canary prompt IDs supplied by A."""
    cdfs = {}
    for slot, prompt_id in requested_config_canary_prompts(requested_key).items():
        logits = load_logits(actual_key, prompt_id)
        probs = make_random_decoding_probability_vector(logits, params)
        cdfs[slot] = make_cumulative_probability_vector(probs, interest_tokens=INTEREST_TOKENS)
    return cdfs


def final_g_dictionary(g_sequences: dict[int, list[float]]) -> dict[int, float]:
    return {
        slot: float(values[-1]) if values else np.nan
        for slot, values in g_sequences.items()
    }


def final_estimator_values(z, g_sequences) -> dict[str, float]:
    g = final_g_dictionary(g_sequences)
    return {
        "method_i_final": float(naive_method_i_estimator(z, g)),
        "method_ii_final": float(naive_method_ii_estimator(z, g)),
    }


def run_status(context_summaries: dict[int, dict]) -> str:
    issues = [
        f"C_{slot}: {summary['status']}"
        for slot, summary in context_summaries.items()
        if summary["status"] != "sampled"
    ]
    return "; ".join(issues) if issues else "sampled"


## 4. Run Requested-Configuration-Canary Stabilization Trial

For each requested model configuration, all three actual configs are evaluated on the requested config's canary prompts. Defined contexts run for exactly `7,500` one-token samples. Undefined target-token support is preserved as an experiment result.


In [4]:
def run_requested_canary_trial(requested_key: str, actual_key: str) -> dict:
    print("=" * 80)
    print(f"Requested Model Config: {MODELS[requested_key]['label']} | Actual Model Config: {MODELS[actual_key]['label']}")
    print("Canary prompts supplied by requested model configuration:", requested_config_canary_prompts(requested_key))

    decode_seed = SHARED_DECODING_SEED
    params = make_random_decoding_params(seed=decode_seed, tokens_to_bias=INTEREST_TOKENS)
    print("Decoding params:", params)

    z = build_requested_z(requested_key)
    cdfs = build_actual_cdfs_on_requested_canaries(requested_key, actual_key, params)

    g_sequences = {}
    context_summaries = {}
    for slot in (1, 2, 4):
        prompt_id = requested_config_canary_prompts(requested_key)[slot]
        probs_by_category = category_probabilities(cdfs[slot])
        status = target_token_status(probs_by_category)
        print(f"slot C_{slot} prompt {prompt_id} category probabilities:", probs_by_category)

        if status != "sampled":
            print(f"WARNING: C_{slot} {status}. Skipping this context.")
            g_sequences[slot] = []
        else:
            sample_seed = (
                SAMPLING_SEED_BASE
                + 1000 * MODEL_ORDER[requested_key]
                + 100 * MODEL_ORDER[actual_key]
                + slot
            )
            rng = np.random.default_rng(sample_seed)
            g_sequences[slot] = sample_g_for_fixed_iterations(
                cdfs[slot],
                num_samples=FIXED_SAMPLES_PER_CONTEXT,
                rng=rng,
                token_a=TOKEN_A,
                token_b=TOKEN_B,
            )

        final_g = float(g_sequences[slot][-1]) if g_sequences[slot] else np.nan
        context_summaries[slot] = {
            "prompt_id": prompt_id,
            "finite_g_values": len(g_sequences[slot]),
            "final_g": final_g,
            "p_token_a": probs_by_category.get(TOKEN_A, 0.0),
            "p_token_b": probs_by_category.get(TOKEN_B, 0.0),
            "p_other": probs_by_category.get(-1, 0.0),
            "status": status,
            "sample_budget": FIXED_SAMPLES_PER_CONTEXT,
        }
        print(f"slot C_{slot} finite g length:", context_summaries[slot]["finite_g_values"])
        print(f"slot C_{slot} final g:", context_summaries[slot]["final_g"])

    estimator_summary = final_estimator_values(z, g_sequences)
    print("Run status:", run_status(context_summaries))
    print("Final Naive Method I estimator:", estimator_summary["method_i_final"])
    print("Final Naive Method II estimator:", estimator_summary["method_ii_final"])
    return {
        "requested_key": requested_key,
        "actual_key": actual_key,
        "canary_provider_key": requested_key,
        "params": params,
        "z": z,
        "cdfs": cdfs,
        "g_sequences": g_sequences,
        "context_summaries": context_summaries,
        "run_status": run_status(context_summaries),
        "estimator_summary": estimator_summary,
    }


trial_results = {}

for requested_key in REQUESTED_MODEL_CONFIG_KEYS:
    for actual_key in ACTUAL_MODEL_CONFIG_KEYS:
        trial_results[(requested_key, actual_key)] = run_requested_canary_trial(requested_key, actual_key)


Requested Model Config: Qwen2.5-7B-Instruct | Actual Model Config: Qwen2.5-7B-Instruct
Canary prompts supplied by requested model configuration: {1: 1, 2: 2, 4: 3}
Decoding params: RandomDecodingParams(temperature=1.0370942971540018, top_p=0.9425723689529342, logit_bias={15: 2, 16: 1}, seed=20260722)
slot C_1 prompt 1 category probabilities: {15: 0.9080498024358086, 16: 0.09195019756419143, -1: 0.0}


slot C_1 finite g length: 7500
slot C_1 final g: 2.328375762134847
slot C_2 prompt 2 category probabilities: {15: 0.5894245906718125, 16: 0.4105754093281875, -1: 0.0}
slot C_2 finite g length: 7500
slot C_2 final g: 0.3606592065340717
slot C_4 prompt 3 category probabilities: {15: 0.8974790920623604, 16: 0.10252090793763957, -1: 0.0}
slot C_4 finite g length: 7500
slot C_4 final g: 2.1987069375246504
Run status: sampled
Final Naive Method I estimator: -0.0036975334406572635
Final Naive Method II estimator: -0.001333549476840723
Requested Model Config: Qwen2.5-7B-Instruct | Actual Model Config: Qwen2.5-14B-Instruct
Canary prompts supplied by requested model configuration: {1: 1, 2: 2, 4: 3}
Decoding params: RandomDecodingParams(temperature=1.0370942971540018, top_p=0.9425723689529342, logit_bias={15: 2, 16: 1}, seed=20260722)


slot C_1 prompt 1 category probabilities: {15: 0.5, 16: 0.5, -1: 0.0}
slot C_1 finite g length: 7500
slot C_1 final g: -0.01813383023803894
slot C_2 prompt 2 category probabilities: {15: 0.08236937877579752, 16: 0.9176306212242025, -1: 0.0}
slot C_2 finite g length: 7500
slot C_2 final g: -2.4048974895925355
slot C_4 prompt 3 category probabilities: {15: 0.8974790920623604, 16: 0.10252090793763957, -1: 0.0}
slot C_4 finite g length: 7500
slot C_4 final g: 2.186896955297133
Run status: sampled
Final Naive Method I estimator: 0.4296176788132231
Final Naive Method II estimator: -1.0331864772624924
Requested Model Config: Qwen2.5-7B-Instruct | Actual Model Config: Qwen2.5-32B-Instruct
Canary prompts supplied by requested model configuration: {1: 1, 2: 2, 4: 3}
Decoding params: RandomDecodingParams(temperature=1.0370942971540018, top_p=0.9425723689529342, logit_bias={15: 2, 16: 1}, seed=20260722)


slot C_1 prompt 1 category probabilities: {15: 0.23053538685559838, 16: 0.7694646131444016, -1: 0.0}
slot C_1 finite g length: 7500
slot C_1 final g: -1.2098175592533362
slot C_2 prompt 2 category probabilities: {15: 0.1025209079376396, 16: 0.8974790920623604, -1: 0.0}
slot C_2 finite g length: 7500
slot C_2 final g: -2.1898390193928314
slot C_4 prompt 3 category probabilities: {15: 0.2760315838183983, 16: 0.7239684161816017, -1: 0.0}


slot C_4 finite g length: 7500
slot C_4 final g: -0.9391767429308899
Run status: sampled
Final Naive Method I estimator: 0.5415659484176969
Final Naive Method II estimator: -1.1859422454195578
Requested Model Config: Qwen2.5-14B-Instruct | Actual Model Config: Qwen2.5-7B-Instruct
Canary prompts supplied by requested model configuration: {1: 1, 2: 4, 4: 5}
Decoding params: RandomDecodingParams(temperature=1.0370942971540018, top_p=0.9425723689529342, logit_bias={15: 2, 16: 1}, seed=20260722)
slot C_1 prompt 1 category probabilities: {15: 0.9080498024358086, 16: 0.09195019756419143, -1: 0.0}
slot C_1 finite g length: 7500
slot C_1 final g: 2.338303175596125
slot C_2 prompt 4 category probabilities: {15: 1.0, 16: 0.0, -1: 0.0}
slot C_2 finite g length: 0
slot C_2 final g: nan
slot C_4 prompt 5 category probabilities: {15: 1.0, 16: 0.0, -1: 0.0}
slot C_4 finite g length: 0
slot C_4 final g: nan
Run status: C_2: undefined: token 16 has zero probability after decoding; C_4: undefined: token 

slot C_1 prompt 1 category probabilities: {15: 0.5, 16: 0.5, -1: 0.0}
slot C_1 finite g length: 7500
slot C_1 final g: 0.03360316162341803
slot C_2 prompt 4 category probabilities: {15: 0.7694646131444016, 16: 0.23053538685559838, -1: 0.0}
slot C_2 finite g length: 7500
slot C_2 final g: 1.1725188063196494
slot C_4 prompt 5 category probabilities: {15: 0.809435860933174, 16: 0.190564139066826, -1: 0.0}
slot C_4 finite g length: 7500
slot C_4 final g: 1.4267857224746283
Run status: sampled
Final Naive Method I estimator: 0.11431643922956691
Final Naive Method II estimator: 0.1340381748681645
Requested Model Config: Qwen2.5-14B-Instruct | Actual Model Config: Qwen2.5-32B-Instruct
Canary prompts supplied by requested model configuration: {1: 1, 2: 4, 4: 5}
Decoding params: RandomDecodingParams(temperature=1.0370942971540018, top_p=0.9425723689529342, logit_bias={15: 2, 16: 1}, seed=20260722)
slot C_1 prompt 1 category probabilities: {15: 0.23053538685559838, 16: 0.7694646131444016, -1: 0.

slot C_2 finite g length: 7500
slot C_2 final g: 1.415720612080464
slot C_4 prompt 5 category probabilities: {15: 0.5, 16: 0.5, -1: 0.0}
slot C_4 finite g length: 7500
slot C_4 final g: -0.014400248839740115
Run status: sampled
Final Naive Method I estimator: 0.660387404100672
Final Naive Method II estimator: 0.9349240599436319
Requested Model Config: Qwen2.5-32B-Instruct | Actual Model Config: Qwen2.5-7B-Instruct
Canary prompts supplied by requested model configuration: {1: 2, 2: 6, 4: 4}
Decoding params: RandomDecodingParams(temperature=1.0370942971540018, top_p=0.9425723689529342, logit_bias={15: 2, 16: 1}, seed=20260722)
slot C_1 prompt 2 category probabilities: {15: 0.5894245906718125, 16: 0.4105754093281875, -1: 0.0}
slot C_1 finite g length: 7500
slot C_1 final g: 0.3711355272749106
slot C_2 prompt 6 category probabilities: {15: 1.0, 16: 0.0, -1: 0.0}
slot C_2 finite g length: 0
slot C_2 final g: nan
slot C_4 prompt 4 category probabilities: {15: 1.0, 16: 0.0, -1: 0.0}
slot C_4 

slot C_1 prompt 2 category probabilities: {15: 0.08236937877579752, 16: 0.9176306212242025, -1: 0.0}
slot C_1 finite g length: 7500
slot C_1 final g: -2.4084139965178837
slot C_2 prompt 6 category probabilities: {15: 0.4400256105559888, 16: 0.5599743894440112, -1: 0.0}
slot C_2 finite g length: 7500
slot C_2 final g: -0.198516044624526
slot C_4 prompt 4 category probabilities: {15: 0.7694646131444016, 16: 0.23053538685559838, -1: 0.0}
slot C_4 finite g length: 7500
slot C_4 final g: 1.1475693383588537
Run status: sampled
Final Naive Method I estimator: 0.8003575541183497
Final Naive Method II estimator: -0.1588838159289348
Requested Model Config: Qwen2.5-32B-Instruct | Actual Model Config: Qwen2.5-32B-Instruct
Canary prompts supplied by requested model configuration: {1: 2, 2: 6, 4: 4}
Decoding params: RandomDecodingParams(temperature=1.0370942971540018, top_p=0.9425723689529342, logit_bias={15: 2, 16: 1}, seed=20260722)
slot C_1 prompt 2 category probabilities: {15: 0.1025209079376396

slot C_2 finite g length: 7500
slot C_2 final g: 0.7348408641562267
slot C_4 prompt 4 category probabilities: {15: 0.809435860933174, 16: 0.190564139066826, -1: 0.0}
slot C_4 finite g length: 7500
slot C_4 final g: 1.450010175505997
Run status: sampled
Final Naive Method I estimator: 0.003805264546018261
Final Naive Method II estimator: 0.002796263887339112


## 5. Context and Final Estimator Summary

The `canary_provider` column should always equal the requested config in this model re-routing protocol. Undefined rows indicate that the fixed decoding parameters/top-p cutoff made at least one target token unsampleable for that context.


In [5]:
context_rows = []
estimator_rows = []

for (requested_key, actual_key), result in trial_results.items():
    for slot in (1, 2, 4):
        summary = result["context_summaries"][slot]
        context_rows.append({
            "A": result["requested_key"],
            "B": actual_key,
            "canary_provider": result["canary_provider_key"],
            "slot": slot,
            "prompt_id": summary["prompt_id"],
            "finite_g_values": summary["finite_g_values"],
            "final_g": summary["final_g"],
            "p_token_a": summary["p_token_a"],
            "p_token_b": summary["p_token_b"],
            "p_other": summary["p_other"],
            "status": summary["status"],
            "sample_budget": summary["sample_budget"],
        })

    estimator_rows.append({
        "A": result["requested_key"],
        "B": actual_key,
        "canary_provider": result["canary_provider_key"],
        "method_i_final": result["estimator_summary"]["method_i_final"],
        "method_ii_final": result["estimator_summary"]["method_ii_final"],
        "status": result["run_status"],
        "expected": "near 0" if result["requested_key"] == actual_key else "nonzero",
    })

print("Context stabilization rows:")
for row in context_rows:
    print(row)

print("\nFinal estimator rows:")
for row in estimator_rows:
    print(row)


Context stabilization rows:
{'A': '7b', 'B': '7b', 'canary_provider': '7b', 'slot': 1, 'prompt_id': 1, 'finite_g_values': 7500, 'final_g': 2.328375762134847, 'p_token_a': 0.9080498024358086, 'p_token_b': 0.09195019756419143, 'p_other': 0.0, 'status': 'sampled', 'sample_budget': 7500}
{'A': '7b', 'B': '7b', 'canary_provider': '7b', 'slot': 2, 'prompt_id': 2, 'finite_g_values': 7500, 'final_g': 0.3606592065340717, 'p_token_a': 0.5894245906718125, 'p_token_b': 0.4105754093281875, 'p_other': 0.0, 'status': 'sampled', 'sample_budget': 7500}
{'A': '7b', 'B': '7b', 'canary_provider': '7b', 'slot': 4, 'prompt_id': 3, 'finite_g_values': 7500, 'final_g': 2.1987069375246504, 'p_token_a': 0.8974790920623604, 'p_token_b': 0.10252090793763957, 'p_other': 0.0, 'status': 'sampled', 'sample_budget': 7500}
{'A': '7b', 'B': '14b', 'canary_provider': '7b', 'slot': 1, 'prompt_id': 1, 'finite_g_values': 7500, 'final_g': -0.01813383023803894, 'p_token_a': 0.5, 'p_token_b': 0.5, 'p_other': 0.0, 'status': 'sam

## 6. Estimator Convergence Graphs

These plots treat the three context traces as parallel sequences. At parallel step `t`, the estimator uses `g_1[t]`, `g_2[t]`, and `g_4[t]`. The x-axis reports total one-token calls across the three contexts, so the plotted x-value is `3t`. Fully undefined curves are omitted from the graph legend; their status remains visible in the printed summary rows above.


In [6]:
from IPython.display import HTML, display

SLOTS = (1, 2, 4)
METHODS = {
    "Naive Method I": naive_method_i_estimator,
    "Naive Method II": naive_method_ii_estimator,
}

COLORS = {
    "7b": "#2563eb",
    "14b": "#dc2626",
    "32b": "#16a34a",
}


def parallel_estimator_curve(
    result: dict,
    estimator_fn,
) -> tuple[np.ndarray, np.ndarray]:
    """Build an estimator trajectory from parallel context iterations.

    At iteration t, the estimator uses g_1[t], g_2[t], and g_4[t]. The x-axis
    is therefore the number of sampling iterations per context, not an
    interleaved or summed count across contexts.
    """
    g_sequences = result["g_sequences"]
    lengths = {slot: len(g_sequences[slot]) for slot in SLOTS}
    if any(lengths[slot] == 0 for slot in SLOTS):
        return np.array([], dtype=np.float64), np.array([], dtype=np.float64)

    curve_length = min(lengths.values())
    xs = np.arange(1, curve_length + 1, dtype=np.float64) * len(SLOTS)
    ys = []

    for index in range(curve_length):
        current_g = {
            slot: float(g_sequences[slot][index])
            for slot in SLOTS
        }
        ys.append(float(estimator_fn(result["z"], current_g)))

    return xs, np.asarray(ys, dtype=np.float64)


def _nice_number(value: float) -> str:
    value = float(value)
    if abs(value) >= 1000 or (abs(value) < 0.001 and value != 0.0):
        return f"{value:.2e}"
    if abs(value) >= 10:
        return f"{value:.1f}"
    return f"{value:.3f}"


def _polyline_points(xs, ys, x_min, x_max, y_min, y_max, left, top, plot_w, plot_h) -> str:
    finite = np.isfinite(xs) & np.isfinite(ys)
    xs = xs[finite]
    ys = ys[finite]
    if len(xs) == 0:
        return ""

    x_span = max(float(x_max - x_min), 1.0)
    y_span = max(float(y_max - y_min), 1e-12)
    px = left + ((xs - x_min) / x_span) * plot_w
    py = top + (1.0 - ((ys - y_min) / y_span)) * plot_h
    return " ".join(f"{x:.2f},{y:.2f}" for x, y in zip(px, py))


def model_label(model_key: str) -> str:
    return MODELS[model_key]["label"]


def legend_label_for_result(result: dict) -> str:
    actual_key = result["actual_key"]
    actual_label = model_label(actual_key)
    if result["run_status"] == "sampled":
        return actual_label
    issue_slots = []
    for slot, summary in result["context_summaries"].items():
        if summary["status"] != "sampled":
            issue_slots.append(f"C_{slot}")
    issue_text = ",".join(issue_slots) if issue_slots else "undefined"
    return f"{actual_label} (undefined: {issue_text})"


def render_svg_plot(requested_key: str, method_name: str, curves: dict[str, tuple[np.ndarray, np.ndarray]]) -> str:
    width = 930
    height = 440
    left = 78
    right = 165
    top = 46
    bottom = 64
    plot_w = width - left - right
    plot_h = height - top - bottom

    x_arrays = [xy[0] for xy in curves.values() if len(xy[0])]
    y_arrays = [xy[1][np.isfinite(xy[1])] for xy in curves.values() if len(xy[1]) and np.any(np.isfinite(xy[1]))]
    has_finite_data = bool(x_arrays) and bool(y_arrays)

    if has_finite_data:
        all_x = np.concatenate(x_arrays)
        all_y = np.concatenate(y_arrays)
        x_min = 0.0
        x_max = float(np.max(all_x))

        final_values = np.array([
            xy[1][np.isfinite(xy[1])][-1]
            for xy in curves.values()
            if len(xy[1]) and np.any(np.isfinite(xy[1]))
        ], dtype=np.float64)
        display_y = all_y
        if len(display_y) > 20:
            low = float(np.quantile(display_y, 0.01))
            high = float(np.quantile(display_y, 0.99))
            display_y = display_y[(display_y >= low) & (display_y <= high)]
            display_y = np.concatenate([display_y, final_values, np.array([0.0])])

        y_min = float(np.min(display_y))
        y_max = float(np.max(display_y))
        if y_min == y_max:
            y_min -= 1.0
            y_max += 1.0
        pad = 0.08 * (y_max - y_min)
        y_min -= pad
        y_max += pad
        y_min = min(y_min, 0.0)
        y_max = max(y_max, 0.0)
    else:
        x_min = 0.0
        x_max = float(FIXED_SAMPLES_PER_CONTEXT * len(SLOTS))
        y_min = -1.0
        y_max = 1.0

    def x_coord(x):
        return left + ((x - x_min) / max(x_max - x_min, 1.0)) * plot_w

    def y_coord(y):
        return top + (1.0 - ((y - y_min) / max(y_max - y_min, 1e-12))) * plot_h

    zero_y = y_coord(0.0)
    svg = [
        f'<svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg" style="max-width:100%; height:auto; font-family: Arial, sans-serif;">',
        f'<rect x="0" y="0" width="{width}" height="{height}" fill="white"/>',
        f'<text x="{width/2:.1f}" y="24" text-anchor="middle" font-size="18" font-weight="700">Estimator Convergence for {method_name} [Requested Model Config: {model_label(requested_key)}]</text>',
        f'<rect x="{left}" y="{top}" width="{plot_w}" height="{plot_h}" fill="#fafafa" stroke="#d4d4d8"/>',
    ]

    if not has_finite_data:
        svg.append(f'<text x="{left + plot_w/2:.1f}" y="{top + plot_h/2:.1f}" text-anchor="middle" font-size="15" fill="#991b1b">No finite estimator curves; see undefined legend entries.</text>')

    for frac in np.linspace(0, 1, 5):
        x = left + frac * plot_w
        x_value = x_min + frac * (x_max - x_min)
        svg.append(f'<line x1="{x:.2f}" y1="{top}" x2="{x:.2f}" y2="{top+plot_h}" stroke="#e5e7eb"/>')
        svg.append(f'<text x="{x:.2f}" y="{top+plot_h+24}" text-anchor="middle" font-size="12" fill="#374151">{int(round(x_value))}</text>')

    for frac in np.linspace(0, 1, 5):
        y = top + frac * plot_h
        y_value = y_max - frac * (y_max - y_min)
        svg.append(f'<line x1="{left}" y1="{y:.2f}" x2="{left+plot_w}" y2="{y:.2f}" stroke="#e5e7eb"/>')
        svg.append(f'<text x="{left-10}" y="{y+4:.2f}" text-anchor="end" font-size="12" fill="#374151">{_nice_number(y_value)}</text>')

    final_value_rows = []
    for actual_key, (xs, ys) in curves.items():
        color = COLORS.get(actual_key, "#525252")
        points = _polyline_points(xs, np.clip(ys, y_min, y_max), x_min, x_max, y_min, y_max, left, top, plot_w, plot_h)
        if points:
            svg.append(f'<polyline points="{points}" fill="none" stroke="{color}" stroke-width="2"/>')
        if len(xs):
            finite = np.isfinite(ys)
            if np.any(finite):
                last_x = xs[finite][-1]
                raw_last_y = float(ys[finite][-1])
                last_y = float(np.clip(raw_last_y, y_min, y_max))
                svg.append(f'<circle cx="{x_coord(last_x):.2f}" cy="{y_coord(last_y):.2f}" r="3.5" fill="{color}"/>')
                final_value_rows.append((actual_key, color, raw_last_y))

    zero_label_x = left + 8
    zero_label_y = min(max(zero_y - 7, top + 14), top + plot_h - 6)
    svg.append(f'<line x1="{left}" y1="{zero_y:.2f}" x2="{left+plot_w}" y2="{zero_y:.2f}" stroke="#111827" stroke-dasharray="6,5" stroke-width="1.75" opacity="0.95"/>')
    svg.append(f'<rect x="{zero_label_x-3:.2f}" y="{zero_label_y-13:.2f}" width="78" height="17" fill="white" opacity="0.86"/>')
    svg.append(f'<text x="{zero_label_x}" y="{zero_label_y:.2f}" text-anchor="start" font-size="12" font-weight="700" fill="#111827">0 baseline</text>')

    panel_x = left + plot_w + 12
    legend_x = panel_x
    legend_y = top + 16
    legend_entries = [
        actual_key
        for actual_key in ACTUAL_MODEL_CONFIG_KEYS
        if len(curves[actual_key][0]) and np.any(np.isfinite(curves[actual_key][1]))
    ]
    svg.append(f'<text x="{legend_x}" y="{legend_y-16}" font-size="12" font-weight="700" fill="#111827">Actual Model Config</text>')
    for idx, actual_key in enumerate(legend_entries):
        y = legend_y + idx * 22
        color = COLORS.get(actual_key, "#525252")
        label = model_label(actual_key)
        svg.append(f'<line x1="{legend_x}" y1="{y}" x2="{legend_x+22}" y2="{y}" stroke="{color}" stroke-width="3"/>')
        svg.append(f'<text x="{legend_x+28}" y="{y+4}" font-size="11" fill="#111827">{label}</text>')

    final_x = panel_x
    final_y = legend_y + 22 * max(len(legend_entries), 1) + 24
    svg.append(f'<text x="{final_x}" y="{final_y}" font-size="12" font-weight="700" fill="#111827">Final Value</text>')
    for idx, (actual_key, color, raw_last_y) in enumerate(final_value_rows):
        y = final_y + 18 + idx * 30
        svg.append(f'<text x="{final_x}" y="{y}" font-size="10.5" fill="{color}">{model_label(actual_key)}:</text>')
        svg.append(f'<text x="{final_x}" y="{y+13}" font-size="10.5" fill="{color}">{raw_last_y:.4g}</text>')

    svg.append(f'<text x="{left + plot_w/2:.1f}" y="{height-16}" text-anchor="middle" font-size="13" fill="#374151">Total LLM calls</text>')
    svg.append(f'<text x="18" y="{top + plot_h/2:.1f}" transform="rotate(-90 18 {top + plot_h/2:.1f})" text-anchor="middle" font-size="13" fill="#374151">Estimator value</text>')
    svg.append('</svg>')
    return "\n".join(svg)


ASSET_DIR = PROJECT_ROOT / "assets"
ASSET_DIR.mkdir(exist_ok=True)


def _slug(text: str) -> str:
    return "_".join(
        "".join(ch.lower() if ch.isalnum() else " " for ch in text).split()
    )


def save_exact_svg_and_png(svg_text: str, output_stem: Path) -> None:
    """Save the exact displayed SVG and a PNG rasterization of that same SVG."""
    import re
    import subprocess
    import tempfile

    svg_path = output_stem.with_suffix('.svg')
    png_path = output_stem.with_suffix('.png')
    svg_path.write_text(svg_text, encoding='utf-8')

    size_match = re.search(r'<svg width="([0-9.]+)" height="([0-9.]+)"', svg_text)
    width = int(float(size_match.group(1))) if size_match else 930
    height = int(float(size_match.group(2))) if size_match else 440

    browser_candidates = [
        Path(r'C:\Program Files\Google\Chrome\Application\chrome.exe'),
        Path(r'C:\Program Files (x86)\Google\Chrome\Application\chrome.exe'),
        Path(r'C:\Program Files (x86)\Microsoft\Edge\Application\msedge.exe'),
        Path(r'C:\Program Files\Microsoft\Edge\Application\msedge.exe'),
    ]
    browser = next((candidate for candidate in browser_candidates if candidate.exists()), None)

    if browser is not None:
        html = f'''<!doctype html>
<html>
<head>
<meta charset="utf-8">
<style>
html, body {{ margin: 0; padding: 0; width: {width}px; height: {height}px; overflow: hidden; background: white; }}
svg {{ display: block; }}
</style>
</head>
<body>{svg_text}</body>
</html>'''
        with tempfile.NamedTemporaryFile('w', suffix='.html', delete=False, encoding='utf-8') as handle:
            html_path = Path(handle.name)
            handle.write(html)
        try:
            subprocess.run(
                [
                    str(browser),
                    '--headless=new',
                    '--disable-gpu',
                    '--hide-scrollbars',
                    '--force-device-scale-factor=2',
                    f'--window-size={width},{height}',
                    f'--screenshot={png_path}',
                    html_path.as_uri(),
                ],
                check=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
            )
            return
        finally:
            html_path.unlink(missing_ok=True)

    try:
        import cairosvg
    except ModuleNotFoundError:
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'cairosvg'])
        import cairosvg

    cairosvg.svg2png(
        bytestring=svg_text.encode('utf-8'),
        write_to=str(png_path),
        scale=2.0,
    )


NOTEBOOK_EXPORT_PREFIX = "model_routing_convergence"

html_parts = []
exported_pngs = []
for requested_key in REQUESTED_MODEL_CONFIG_KEYS:
    for method_name, estimator_fn in METHODS.items():
        curves = {
            actual_key: parallel_estimator_curve(
                trial_results[(requested_key, actual_key)], estimator_fn
            )
            for actual_key in ACTUAL_MODEL_CONFIG_KEYS
        }
        svg_text = render_svg_plot(requested_key, method_name, curves)
        html_parts.append(svg_text)
        output_stem = ASSET_DIR / f"{NOTEBOOK_EXPORT_PREFIX}_requested_config_{requested_key}_{_slug(method_name)}"
        save_exact_svg_and_png(svg_text, output_stem)
        exported_pngs.append(output_stem.with_suffix('.png'))

display(HTML("<div>" + "<hr style='margin:24px 0;'>".join(html_parts) + "</div>"))
print("Saved exact SVG/PNG exports:")
for path in exported_pngs:
    print(path)


Saved exact SVG/PNG exports:
C:\Users\beatn\LLM-Ownership-Identification-Draft-1\assets\model_routing_convergence_requested_config_7b_naive_method_i.png
C:\Users\beatn\LLM-Ownership-Identification-Draft-1\assets\model_routing_convergence_requested_config_7b_naive_method_ii.png
C:\Users\beatn\LLM-Ownership-Identification-Draft-1\assets\model_routing_convergence_requested_config_14b_naive_method_i.png
C:\Users\beatn\LLM-Ownership-Identification-Draft-1\assets\model_routing_convergence_requested_config_14b_naive_method_ii.png
C:\Users\beatn\LLM-Ownership-Identification-Draft-1\assets\model_routing_convergence_requested_config_32b_naive_method_i.png
C:\Users\beatn\LLM-Ownership-Identification-Draft-1\assets\model_routing_convergence_requested_config_32b_naive_method_ii.png
